In [ ]:
from jira import JIRA

from datetime import datetime,timedelta
import sys
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
import numpy as np
import pandas as pd
import json

In [ ]:
import logging
def get_logger():
    logger = logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    #fh = logging.FileHandler(
    #    "E:/wangzhilin/QuantumGalaxy/logs/flask_server.log",
    #    'a',
    #    encoding='utf-8')
    #fh.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    formatter = logging.Formatter(
        fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
        datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    #fh.setFormatter(formatter)
    logger.addHandler(ch)
    #logger.addHandler(fh)
    return logger
logger=get_logger()

In [ ]:
code='688063.SH'


In [ ]:
import scipy,math
def log_inter1(num:int,y0,y1):
    '''返回长度为num+2的插值列表。num:插值数量'''
    f=scipy.interpolate.interp1d(x=[0,num+1],y=[math.log(y0),math.log(y1)],kind='linear')
    res=f([x+1 for x in range(num)])

    res=[math.exp(r) for r in res]
    res.insert(0,y0)
    res.append(y1)
    return res
def geo_mean(data):
    return math.pow(np.prod(data),1.0/len(data))
def calcu_norm(xc,pc,xo,po,ppf) :
    '''-> (s,mu)'''
    xc=np.log(xc)
    xo=np.log(xo)
    no=round((1-po)*10-1)
    nc=round((1-pc)*10-1)
    #print(no,nc)
    s=(xo-xc)/(ppf[no]-ppf[nc])
    mu=((xo-ppf[no]*s+xc-ppf[nc]*s)/2)
    return(s,mu)

In [ ]:
xo=issue.fields.customfield_10918
po=issue.fields.customfield_10919
xc=issue.fields.customfield_10916
pc=issue.fields.customfield_10914
ppf = [-1.282, -0.842, -0.524, -0.253, 0, 0.253, 0.524, 0.842, 1.282]
calcu_norm(xc=xc,xo=xo,pc=pc,po=po,ppf=ppf)

In [ ]:
def data_process(code):
    host='localhost'
    user='Local_Editor'
    password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(host,user,password,database)
    jira= JIRA('https://research.quantumgalaxy.cn/', basic_auth=('wangzhilin', "wangzhilin"))
    try:
        issue=jira.search_issues(f'project = POSITION AND status != 尚未纳入 AND resolution = Unresolved AND cf[10201] ~  {code}')[0]
    except:
        logger.error(f'no position issue, code: {code}')
        return False
    target_date=issue.fields.customfield_10622
    start_date=issue.fields.updated[:10]

    xo=issue.fields.customfield_10918
    po=issue.fields.customfield_10919
    xc=issue.fields.customfield_10916
    pc=issue.fields.customfield_10914
    
    stock_start_date=datetime.strptime(start_date,'%Y-%m-%d')-timedelta(183)
    stock_start_date=stock_start_date.strftime('%Y-%m-%d')


    ev_data=mysql.read_query('select date,market_value from ticker_data where code="%s" and date<=now() and date>="%s"'%(code,stock_start_date))
    ev_data=[ev for ev in ev_data]
    stock_end_date=ev_data[-1][0]

    import math


    ppf = [-1.282, -0.842, -0.524, -0.253, 0, 0.253, 0.524, 0.842, 1.282]


    s,mu=calcu_norm(xc=xc,xo=xo,pc=pc,po=po,ppf=ppf)
    #print(s,mu)
    df=pd.DataFrame(data=ev_data,columns=['date','ev'])
    df.set_index('date',inplace=True)


    stock_data=mysql.read_query(f'select date,close_price from ticker_data where code="{code}" and date<="{start_date}" order by date desc limit 60')
    vol=mysql.read_query(f'select vol from processed_data where code="{code}" and date<="{start_date}" order by date desc limit 1')
    vol=vol[0][0]
    stock_data=[float(p) for (d,p) in stock_data]
    stock_data=geo_mean(stock_data)

    factor=[(1-vol)**10,(1-vol)**5,1,(1+vol)**5,(1+vol)**10,]
    start=np.outer(stock_data,factor)[0]
    n=mysql.read_query('select value from customized_data where code="C00002.QG" order by date desc limit 1')[0][0]
    end=np.exp([mu-0.842*s,mu-0.5*s,mu,mu+0.5*s,mu+0.842*s])*n

    start_date=datetime.strptime(start_date,'%Y-%m-%d')
    target_date=datetime.strptime(target_date,'%Y-%m-%d')
    days=(target_date-start_date).days
    interp_num=days-1
    day_list=[start_date+timedelta(i) for i in range(days+1)]
    day_list=[datetime.date(d) for d in day_list]
    level0=log_inter1(interp_num,start[0],end[0])

    level1=log_inter1(interp_num,start[1],end[1])

    level2=log_inter1(interp_num,start[2],end[2])

    level3=log_inter1(interp_num,start[3],end[3])

    level4=log_inter1(interp_num,start[4],end[4])
    level4=[level4[i]-level3[i] for i in range(len(level4))]
    level3=[level3[i]-level2[i] for i in range(len(level4))]
    level2=[level2[i]-level1[i] for i in range(len(level4))]
    level1=[level1[i]-level0[i] for i in range(len(level4))]
    level0=tuple(dict(zip(day_list,level0)).items())
    level1=tuple(dict(zip(day_list,level1)).items())
    level2=tuple(dict(zip(day_list,level2)).items())
    level3=tuple(dict(zip(day_list,level3)).items())
    level4=tuple(dict(zip(day_list,level4)).items())
    down_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1])
    pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1])
    up_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1]+level3[-1][1])
    down_pred_point=np.round(down_pred_point[1],2)
    pred_point=np.round(pred_point[1],2)

    up_pred_point=np.round(up_pred_point[1],2)
    df['last_ev']=None
    df['last_ev'][-1]=df['ev'][-1]
    df1=pd.DataFrame(level0,columns=['date','level0'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level1,columns=['date','level1'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level2,columns=['date','level2'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level3,columns=['date','level3'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df1=pd.DataFrame(level4,columns=['date','level4'])
    df1.set_index('date',inplace=True)
    df=pd.concat([df,df1],axis=1)
    df['down_pred_point']=None
    df['pred_point']=None
    df['up_pred_point']=None
    down_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1])
    pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1])
    up_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1]+level3[-1][1])
    down_pred_point=np.round(down_pred_point[1],2)
    pred_point=np.round(pred_point[1],2)

    up_pred_point=np.round(up_pred_point[1],2)
    
    df.index=df.index.astype('str',)
    df=df.reset_index()
    df.loc[df.shape[0]-1,'down_pred_point']=down_pred_point
    df.loc[df.shape[0]-1,'up_pred_point']=up_pred_point
    df.loc[df.shape[0]-1,'pred_point']=pred_point
    df = df.astype(object).replace(np.nan, 'None')
    dataset={}
    dataset['source']=[df.columns.to_list()]
    dataset['source']
    for i,r in df.iterrows():

        row=r.to_list()
        dataset['source'].append(row)
    j=json.dumps(dataset)
    return j

In [ ]:
issue.fields.__dict__

In [ ]:
jira= JIRA('https://research.quantumgalaxy.cn/', basic_auth=('wangzhilin', "wangzhilin"))
try:
    issue=jira.search_issues(f'project = POSITION AND status != 尚未纳入 AND resolution = Unresolved AND cf[10201] ~  {code}')[0]
except:
    logger.error(f'no position issue, code: {code}')

target_date=issue.fields.customfield_10622
start_date=issue.fields.updated[:10]

xo=issue.fields.customfield_10918
po=issue.fields.customfield_10919
xc=issue.fields.customfield_10916
pc=issue.fields.customfield_10914

In [ ]:
from atlassian import Jira

jira = Jira(
    url='https://research.quantumgalaxy.cn/',
    username='wangzhilin',
    password='wangzhilin')


In [278]:
code='600256'

In [279]:
'''fields['customfield_11206']
10801: m=x
    10800 m=xn+a
'''

'10801: m=x\n'

In [280]:
issue=jira.jql(f'project = POSITION  AND resolution = Unresolved AND cf[10201] ~  {code}')

In [287]:
model=issue['issues'][0]['fields']['customfield_11206']
print(model)
a= 'xn' if model['id']=='10800'  else 'x'
a

{'self': 'https://research.quantumgalaxy.cn/rest/api/2/customFieldOption/10800', 'value': 'M=Xn+A模式', 'id': '10800', 'disabled': False}


'xn'

In [276]:
model['id']

'10801'

In [212]:
issue['issues'][0]['key']

'POSITION-3900'

In [213]:
log=jira.get_issue_changelog('POSITION-2969')['histories']

In [214]:
log

[{'id': '114269',
  'author': {'self': 'https://research.quantumgalaxy.cn/rest/api/2/user?username=bot',
   'name': 'bot',
   'key': 'JIRAUSER10200',
   'emailAddress': 'jira_bot@quantumgalaxy.cn',
   'avatarUrls': {'48x48': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=48',
    '24x24': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=24',
    '16x16': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=16',
    '32x32': 'https://www.gravatar.com/avatar/3d75f49b419a66aa39f16798646dd968?d=mm&s=32'},
   'displayName': '研究机器人',
   'active': True,
   'timeZone': 'Asia/Shanghai'},
  'created': '2021-10-04T12:45:44.000+0800',
  'items': [{'field': '万得代码',
    'fieldtype': 'custom',
    'from': None,
    'fromString': None,
    'to': None,
    'toString': '688063.SH'}]},
 {'id': '114270',
  'author': {'self': 'https://research.quantumgalaxy.cn/rest/api/2/user?username=bot',
   'name': 'bot',
   'key': 'JIRAUSER10200',
  

In [223]:
row=log[0]

row['items']

[{'field': '万得代码',
  'fieldtype': 'custom',
  'from': None,
  'fromString': None,
  'to': None,
  'toString': '688063.SH'}]

In [216]:

delta=datetime.strptime(row1['created'][:16],'%Y-%m-%dT%H:%M')-datetime.strptime(row['created'][:16],'%Y-%m-%dT%H:%M')


In [217]:
issue['issues'][0]['fields']['summary']

'600536.SH 中国软件'

In [224]:
def log_inter1(num:int,y0,y1):
    '''返回长度为num+2的插值列表。num:插值数量'''
    f=scipy.interpolate.interp1d(x=[0,num+1],y=[math.log(y0),math.log(y1)],kind='linear')
    res=f([x+1 for x in range(num)])

    res=[math.exp(r) for r in res]
    res.insert(0,y0)
    res.append(y1)
    return res
def geo_mean(data):
    return math.pow(np.prod(data),1.0/len(data))
def calcu_norm(xc,pc,xo,po,ppf) :
    '''-> (s,mu)'''
    xc=np.log(xc)
    xo=np.log(xo)
    no=round((1-po)*10-1)
    nc=round((1-pc)*10-1)
    #print(no,nc)
    s=(xo-xc)/(ppf[no]-ppf[nc])
    mu=((xo-ppf[no]*s+xc-ppf[nc]*s)/2)
    return(s,mu)

In [252]:
issue['issues'][0]['fields']['customfield_10201']

'600536.SH'

In [255]:
import sys
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL


class Code2JS():
    '''对issue changelog分析并拆解为多个走廊（一个走廊初始数据为6个数）'''
    def __init__(self,code):
        self.code=code
        host='localhost'
        user='Local_Editor'
        password='QuantumGalaxy'
        database='qgdbs'
        self.mysql=MYSQL(host,user,password,database)
    def find_issue(self,code):
        from atlassian import Jira

        jira = Jira(
            url='https://research.quantumgalaxy.cn/',
            username='wangzhilin',
            password='wangzhilin')
        self.issue=jira.jql(f'project = POSITION  AND resolution = Unresolved AND cf[10201] ~  {code}')
        key=self.issue['issues'][0]['key']
        self.code=self.issue['issues'][0]['fields']['customfield_10201']
        log=jira.get_issue_changelog(key)['histories']
        return log
    def fetch_summary(self):
        return self.issue['issues'][0]['fields']['summary']
    def clean_data(self,log):
        '''只保留issue changelog中包含 目标情景日/xo/po/xc/pc等字段的日志及其时间'''
        history=[]
        for row in log:
            t=datetime.strptime(row['created'][:16],'%Y-%m-%dT%H:%M')
            items=row['items']
            new={}
            get=False
            for item in items:
                field=item['field']
                if field  in ['未来情景日','保守认为利润(Xc)','乐观认为利润(Xo)','乐观情景概率','保守情景概率']:
                    
                    new['start_date']=t
                    if field=='未来情景日':
                        new['target_date']=datetime.strptime(item['toString'],'%Y-%m-%d')
                    if field=='保守认为利润(Xc)':
                        new['xc']=item['toString']
                    if field=='乐观认为利润(Xo)':
                        new['xo']=item['toString']
                    if field=='乐观情景概率':
                        new['po']=item['toString']
                    if field=='保守情景概率':
                        new['pc']=item['toString']



                    if not get:
                        get=True
                        history.append(new)

                    
                       

        return history
    def split_ev_band(self,history):
        '''以12小时为界，间隔低于12小时则不视为新走廊；将所有走廊的完整数据生成'''
        '''issue:不视为新走廊，但数值应当每次更新为最新'''
        results=[]
        for i in range(len(history)):
            
            if i ==0 :
                start_date=history[i].get('start_date',None)
                target_date=history[i].get('target_date',None)
                pc=history[i].get('pc',None)
                po=history[i].get('po',None)
                xc=history[i].get('xc',None)
                xo=history[i].get('xo',None)
                result=history[i]
                keys=['start_date','target_date','pc','xc','xo','po']
                values=[start_date,target_date,pc,xc,xo,po]

            
               
            start_date=history[i].get('start_date',start_date)
            target_date=history[i].get('target_date',target_date)
            pc=history[i].get('pc',pc)
            po=history[i].get('po',po)
            xc=history[i].get('xc',xc)
            xo=history[i].get('xo',xo)
            values=[start_date,target_date,pc,xc,xo,po]
            result=dict(zip(keys,values))
            #print(result)
            if i<len(history)-1:
                delta=history[i+1]['start_date']-history[i]['start_date']
                if delta.days==0 and delta.seconds<43200:
                        pass
                else:
                    
                    results.append(result)
            else:
                results.append(result)
        return results
    def work(self):
        log=self.find_issue(self.code)                    
        history=self.clean_data(log)
        results=self.split_ev_band(history)
        js=[]
        for r in results:
            js.append(self.one_work(r))

        return js
    def one_work(self,data):
        mysql=self.mysql
        code=self.code
        try:
            #print(data)
            target_date=data['target_date']
            start_date=data['start_date']
            
            xo,xc,po,pc=float(data['xo']),float(data['xc']),float(data['po']),float(data['pc'])
            
            stock_start_date=start_date-timedelta(183)
            stock_start_date=stock_start_date.strftime('%Y-%m-%d')


            ev_data=mysql.read_query('select date,market_value from ticker_data where code="%s" and date<=now() and date>="%s"'%(code,stock_start_date))
            ev_data=[ev for ev in ev_data]
            stock_end_date=ev_data[-1][0]

            import math


            ppf = [-1.282, -0.842, -0.524, -0.253, 0, 0.253, 0.524, 0.842, 1.282]


            s,mu=calcu_norm(xc,pc,xo,po,ppf)
            #print(s,mu)
            df=pd.DataFrame(data=ev_data,columns=['date','ev'])
            df.set_index('date',inplace=True)


            stock_data=mysql.read_query(f'select date,market_value from ticker_data where code="{code}" and date<="{start_date}" order by date desc limit 60')
            vol=mysql.read_query(f'select vol from processed_data where code="{code}" and date<="{start_date}" order by date desc limit 1')
            vol=vol[0][0]

            stock_data=[float(p) for (d,p) in stock_data]
            stock_data=geo_mean(stock_data)
            #print(stock_data)

            factor=[(1-vol)**10,(1-vol)**5,1,(1+vol)**5,(1+vol)**10,]
            start=np.outer(stock_data,factor)[0]
            #print(start)
            n=mysql.read_query('select value from customized_data where code="C00002.QG" order by date desc limit 1')[0][0]
            end=np.exp([mu-0.842*s,mu-0.5*s,mu,mu+0.5*s,mu+0.842*s])*n
            
            #print(start_date)
            #start_date=datetime.strptime(start_date,'%Y-%m-%d')
            #target_date=datetime.strptime(target_date,'%Y-%m-%d')
            days=(target_date-start_date).days
            interp_num=days-1
            day_list=[start_date+timedelta(i) for i in range(days+1)]
            day_list=[datetime.date(d) for d in day_list]
            level0=log_inter1(interp_num,start[0],end[0])

            level1=log_inter1(interp_num,start[1],end[1])

            level2=log_inter1(interp_num,start[2],end[2])

            level3=log_inter1(interp_num,start[3],end[3])

            level4=log_inter1(interp_num,start[4],end[4])
            level4=[level4[i]-level3[i] for i in range(len(level4))]
            level3=[level3[i]-level2[i] for i in range(len(level4))]
            level2=[level2[i]-level1[i] for i in range(len(level4))]
            level1=[level1[i]-level0[i] for i in range(len(level4))]
            level0=tuple(dict(zip(day_list,level0)).items())
            level1=tuple(dict(zip(day_list,level1)).items())
            level2=tuple(dict(zip(day_list,level2)).items())
            level3=tuple(dict(zip(day_list,level3)).items())
            level4=tuple(dict(zip(day_list,level4)).items())
            down_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1])
            pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1])
            up_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1]+level3[-1][1])
            down_pred_point=np.round(down_pred_point[1],2)
            pred_point=np.round(pred_point[1],2)

            up_pred_point=np.round(up_pred_point[1],2)
            df['last_ev']=None
            df.loc[df.index[-1],'last_ev']=df['ev'][-1]
            #df['last_ev'][-1]=df['ev'][-1]
            df1=pd.DataFrame(level0,columns=['date','level0'])
            df1.set_index('date',inplace=True)
            df=pd.concat([df,df1],axis=1)
            df1=pd.DataFrame(level1,columns=['date','level1'])
            df1.set_index('date',inplace=True)
            df=pd.concat([df,df1],axis=1)
            df1=pd.DataFrame(level2,columns=['date','level2'])
            df1.set_index('date',inplace=True)
            df=pd.concat([df,df1],axis=1)
            df1=pd.DataFrame(level3,columns=['date','level3'])
            df1.set_index('date',inplace=True)
            df=pd.concat([df,df1],axis=1)
            df1=pd.DataFrame(level4,columns=['date','level4'])
            df1.set_index('date',inplace=True)
            df=pd.concat([df,df1],axis=1)
            df['down_pred_point']=None
            df['pred_point']=None
            df['up_pred_point']=None
            down_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1])
            pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1])
            up_pred_point=(level0[-1][0],level0[-1][1]+level1[-1][1]+level2[-1][1]+level3[-1][1])
            down_pred_point=np.round(down_pred_point[1],2)
            pred_point=np.round(pred_point[1],2)

            up_pred_point=np.round(up_pred_point[1],2)
            
            df.index=df.index.astype('str',)
            df=df.reset_index()
            df.loc[df.shape[0]-1,'down_pred_point']=down_pred_point
            df.loc[df.shape[0]-1,'up_pred_point']=up_pred_point
            df.loc[df.shape[0]-1,'pred_point']=pred_point
            df = df.astype(object).replace(np.nan, 'None')
            dataset={}
            dataset['source']=[df.columns.to_list()]
            dataset['source']
            for i,r in df.iterrows():

                row=r.to_list()
                dataset['source'].append(row)
            j=json.dumps(dataset)
            
            return j
        except:

            raise Exception
            logger.error('had exception during process data')
            return False,False
    


a=Code2JS(code)

                    




10-28 17:35 logger       INFO     using username Local_Editor
10-28 17:35 logger       INFO     accessed to database <qgdbs>


In [259]:
r[0]['start_date'].strftime("%Y-%m-%d")

'2022-10-14'

In [193]:
ppf = [-1.282, -0.842, -0.524, -0.253, 0, 0.253, 0.524, 0.842, 1.282]
xo,xc,po,pc=54,13,0.4,0.7


In [195]:
a.one_work(result[0])


10-28 14:43 logger       INFO     got a cursor
10-28 14:43 logger       INFO     Accepted read sql query "select date,market_value from ticker_data where code="688063.SH" and date<=now() and date>="2022-02-23""
10-28 14:43 logger       INFO     got a cursor
10-28 14:43 logger       INFO     Accepted read sql query "select date,market_value from ticker_data where code="688063.SH" and date<="2022-08-25 22:34:00" order by date desc limit 60"
10-28 14:43 logger       INFO     got a cursor
10-28 14:43 logger       INFO     Accepted read sql query "select vol from processed_data where code="688063.SH" and date<="2022-08-25 22:34:00" order by date desc limit 1"
10-28 14:43 logger       INFO     got a cursor
10-28 14:43 logger       INFO     Accepted read sql query "select value from customized_data where code="C00002.QG" order by date desc limit 1"


'{"source": [["date", "ev", "last_ev", "level0", "level1", "level2", "level3", "level4", "down_pred_point", "pred_point", "up_pred_point"], ["2022-02-23", 244.9485667527, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-02-24", 232.9945688051, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-02-25", 240.86067108150002, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-02-28", 229.16990884, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-03-01", 227.63694796329997, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-03-02", 227.49758788359998, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-03-03", 218.9192007554, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-03-04", 215.24938532329998, "None", "None", "None", "None", "None", "None", "None", "None", "None"], ["2022-03-07",

In [154]:
start_date

datetime.datetime(2022, 10, 25, 0, 0)

In [152]:
len(r)

10

In [ ]:
log[1]

In [ ]:
df=data_process(code)


In [ ]:

df